In [ ]:
import os
import re
import logging
import sys
import torch
import torch.nn as nn
from typing import Dict, Tuple
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import subprocess

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

# Log environment information
logger.info("\n" + "=" * 80)
logger.info("ENVIRONMENT CONFIGURATION")
logger.info("=" * 80)
logger.info(f"Python version: {sys.version}")
logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"CUDA version: {torch.version.cuda}")
    logger.info(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        logger.info(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    logger.info("Running on CPU")

try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        logger.info(f"NVIDIA GPU: {result.stdout.strip()}")
except Exception:
    pass
logger.info("=" * 80 + "\n")

04/15/2026 00:59:38 - INFO - __main__ - 
04/15/2026 00:59:38 - INFO - __main__ - ENVIRONMENT CONFIGURATION
04/15/2026 00:59:38 - INFO - __main__ - ================================================================================
04/15/2026 00:59:38 - INFO - __main__ - Python version: 3.12.0 (main, Oct  2 2023, 23:53:49) [MSC v.1929 64 bit (AMD64)]
04/15/2026 00:59:38 - INFO - __main__ - PyTorch version: 2.11.0+cpu
04/15/2026 00:59:38 - INFO - __main__ - CUDA available: False
04/15/2026 00:59:38 - INFO - __main__ - Running on CPU
04/15/2026 00:59:38 - INFO - __main__ - ================================================================================



In [ ]:
!uv pip install --upgrade pip
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision pandas

!uv pip uninstall transformers tokenizers accelerate -q

!uv pip install "transformers==4.56.0" "protobuf==5.29.3" -q
!uv pip install torch datasets -q
!uv pip install tqdm wandb
!uv pip install bitsandbytes accelerate hf_transfer
# !uv pip install -r requirements.txt
!uv pip install --force-reinstall --no-cache-dir "numpy==2.0"

In [ ]:
# ============================================================================
# 1. DATA LAYER: Load and preprocess datasets
# ============================================================================

def preprocess_code(code_str: str) -> str:
    """Normalize code string."""
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)
    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()


def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    """Tokenize code examples."""
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)


def load_datasets(tokenizer, max_length: int = 512) -> Tuple[Dataset, Dataset]:
    """Load and tokenize train and validation datasets."""
    logger.info("Loading datasets from Hugging Face Hub...")

    train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
    val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

    logger.info(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

    # Tokenize
    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train",
    )

    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val",
    )

    # Rename label column
    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")

    return train_dataset, val_dataset

# ============================================================================
# 1.5 METRICS: Evaluation metric function for Trainer
# ============================================================================

def compute_metrics_fn(eval_pred):
    """Compute precision, recall, F1, and accuracy for evaluation."""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }

In [ ]:
# ============================================================================
# 3. TRAINING ENGINE: HF Trainer handles training, validation, and logging
# ============================================================================

# Note: The Trainer class from transformers handles:
# - Training loop with gradient accumulation
# - Validation every N steps (eval_steps=200)
# - Best model selection based on macro_f1
# - Automatic W&B logging if enabled
# - Checkpointing and model saving

In [ ]:
# ============================================================================
# 2. MODEL LAYER: Load pretrained CodeBERT and tokenizer
# ============================================================================

def load_model_and_tokenizer(model_name: str, num_labels: int = 2):
    """Load pretrained model and tokenizer from Hugging Face."""
    logger.info(f"Loading tokenizer from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    logger.info(f"Loading model from: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification",
    )

    logger.info(f"Model loaded successfully")
    logger.info(f"Model config - num_labels: {model.config.num_labels}, hidden_size: {model.config.hidden_size}")

    return model, tokenizer


def freeze_base_model(model):
    """
    Freeze all base model parameters, keeping only classifier head trainable.

    For sequence classification models (e.g., CodeBERT), this freezes the
    RoBERTa/encoder layers and unfreezes the classification head.

    Args:
        model: The pretrained model to freeze
    """
    # Freeze all encoder/base model parameters
    for name, param in model.named_parameters():
        if "classifier" not in name and "cls" not in name.lower():
            param.requires_grad = False

    # Ensure classifier head is trainable
    for name, param in model.named_parameters():
        if "classifier" in name or "cls" in name.lower():
            param.requires_grad = True

    logger.info("Base model frozen - only classifier head is trainable")


In [ ]:
# ============================================================================
# 3. TRAINING ENGINE: HF Trainer handles training, validation, and logging
# ============================================================================

def train_pipeline(
    model_name: str = "microsoft/codebert-base",
    output_dir: str = "./taskA-codebert-base",
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 2e-5,
    max_length: int = 512,
    num_labels: int = 2,
    use_wandb: bool = False,
    freeze_base: bool = False,
    loss_type: str = "ce",
    focal_alpha: float = 1.0,
    focal_gamma: float = 2.0,
    r_drop_alpha: float = 4.0,
    infonce_temperature: float = 0.07,
    infonce_weight: float = 0.5,
):
    from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)

    log_file = os.path.join(output_dir, "training.log")
    file_handler = logging.FileHandler(log_file, mode="w")
    file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(name)s - %(message)s"))
    logger.addHandler(file_handler)
    logger.info(f"Logging to file: {log_file}")

    model, tokenizer = load_model_and_tokenizer(model_name, num_labels)
    model = model.to(device)
    logger.info(f"Model on device: {device}")

    if loss_type == "infonce":
        model.config.output_hidden_states = True
        logger.info("Enabled hidden states output for InfoNCE loss")

    if freeze_base:
        logger.info("Freezing base model weights, training classifier head only...")
        freeze_base_model(model)

    train_dataset, val_dataset = load_datasets(tokenizer, max_length)

    if use_wandb:
        try:
            import wandb
            os.environ["WANDB_MODE"] = "offline"
            logger.info("W&B set to offline mode for notebook stability")
        except Exception as e:
            logger.warning(f"Could not setup W&B: {e}. Continuing without W&B.")
            use_wandb = False

    steps_per_epoch = max(1, len(train_dataset) // batch_size)
    total_steps = num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))

    # load_best_model_at_end=True ensures the model retains the BEST weights at the end
    # of the training process, maintaining strict alignment between training and inference weights.
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        weight_decay=0.01,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        logging_strategy="steps",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to=["wandb"] if use_wandb else [],
        run_name="codebert-base-task-a" if use_wandb else None,
        gradient_checkpointing=True,
        dataloader_pin_memory=True if torch.cuda.is_available() else False,
        seed=42,
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    compute_loss_func = None
    if loss_type == "focal":
        compute_loss_func = lambda outputs, labels, num_items_in_batch=None: focal_loss_compute(
            outputs, labels, num_items_in_batch, alpha=focal_alpha, gamma=focal_gamma
        )
    elif loss_type == "infonce":
        compute_loss_func = lambda outputs, labels, num_items_in_batch=None: infonce_loss_compute(
            outputs, labels, num_items_in_batch, temperature=infonce_temperature, infonce_weight=infonce_weight
        )
    elif loss_type == "ce":
        compute_loss_func = None

    class CustomTrainer(Trainer):
        def __init__(self, *args, compute_loss_func=None, loss_type="ce", r_drop_alpha=4.0, **kwargs):
            super().__init__(*args, **kwargs)
            self.compute_loss_func = compute_loss_func
            self.loss_type = loss_type
            self.r_drop_alpha = r_drop_alpha

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels") if "labels" in inputs else None

            # Correct R-Drop Implementation requiring dual forward passes
            if self.loss_type == "r-drop" and labels is not None and model.training:
                outputs1 = model(**inputs)
                outputs2 = model(**inputs)

                logits1, logits2 = outputs1.logits, outputs2.logits
                ce_loss = (torch.nn.functional.cross_entropy(logits1, labels) +
                           torch.nn.functional.cross_entropy(logits2, labels)) / 2

                p = torch.nn.functional.log_softmax(logits1, dim=-1)
                q = torch.nn.functional.log_softmax(logits2, dim=-1)

                p_loss = torch.nn.functional.kl_div(p, torch.nn.functional.softmax(logits2, dim=-1), reduction='batchmean')
                q_loss = torch.nn.functional.kl_div(q, torch.nn.functional.softmax(logits1, dim=-1), reduction='batchmean')
                kl_loss = (p_loss + q_loss) / 2

                loss = ce_loss + self.r_drop_alpha * kl_loss
                return (loss, outputs1) if return_outputs else loss

            # Standard forward passes for CE, Focal, InfoNCE
            outputs = model(**inputs)

            if labels is not None:
                if self.compute_loss_func is not None and self.loss_type != "r-drop":
                    loss = self.compute_loss_func(outputs, labels, num_items_in_batch=num_items_in_batch)
                else:
                    loss = torch.nn.functional.cross_entropy(outputs.logits, labels)
            else:
                loss = outputs.loss if hasattr(outputs, "loss") else outputs[0]

            return (loss, outputs) if return_outputs else loss

    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        compute_loss_func=compute_loss_func,
        loss_type=loss_type,
        r_drop_alpha=r_drop_alpha
    )

    logger.info("Starting training...")
    trainer.train()

    logger.info(f"\n🎯 Training complete!")
    logger.info(f"Best model weights automatically loaded into trainer.model")
    logger.info(f"Best model saved to: {os.path.join(output_dir, 'best_model')}")

    return trainer, model, tokenizer

# Start training. Feel free to swap loss functions here.
trainer, model, tokenizer = train_pipeline(
    model_name="microsoft/codebert-base",
    output_dir="./taskA-codebert-base",
    num_epochs=3,
    batch_size=32,
    learning_rate=2e-5,
    max_length=512,
    use_wandb=False,
    freeze_base=False,
    loss_type="ce",
    focal_alpha = 1.0,
    focal_gamma = 2.0,
    r_drop_alpha = 4.0,
    infonce_temperature = 0.07,
    infonce_weight = 0.5,
)


In [ ]:
# ============================================================================
# 5. INFERENCE: Generate predictions on test set
# ============================================================================
def inference_pipeline(
    trainer,
    tokenizer,
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
):
    """Generate predictions on test set using the best trained model."""
    logger.info("Loading test dataset...")
    test_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test")

    # Tokenize test set
    logger.info("Tokenizing test dataset...")
    test_dataset = test_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in test_dataset.column_names],
        desc="Tokenizing test",
    )

    # Ensure correct column naming for automatic evaluation
    if "label" in test_dataset.column_names:
        test_dataset = test_dataset.rename_column("label", "labels")

    # Generate predictions using the loaded BEST model from training
    logger.info("Generating predictions on test set using best model weights...")
    predictions = trainer.predict(test_dataset)
    logits = predictions.predictions
    pred_labels = logits.argmax(axis=-1)

    # Output Eval Metrics for Inference (if true labels exist)
    if "labels" in test_dataset.column_names:
        logger.info("\n" + "=" * 50)
        logger.info("TEST SET EVALUATION METRICS")
        logger.info("=" * 50)
        metrics = compute_metrics_fn((logits, test_dataset["labels"]))
        for metric_name, value in metrics.items():
            logger.info(f"{metric_name.capitalize()}: {value:.4f}")
        logger.info("=" * 50 + "\n")

    # Write to CSV
    logger.info(f"Writing predictions to {output_csv}...")
    with open(output_csv, "w") as f:
        f.write("id,label\n")
        for i, pred in enumerate(pred_labels):
            f.write(f"{i},{pred}\n")

    logger.info(f"✅ Predictions saved to {output_csv}")
    logger.info(f"Total predictions: {len(pred_labels)}")
    logger.info(f"Class distribution: {[(pred_labels == i).sum() for i in range(2)]}")

    return pred_labels


logger.info("\n" + "=" * 80)
logger.info("INFERENCE")
logger.info("=" * 80 + "\n")

predictions = inference_pipeline(
    trainer=trainer,
    tokenizer=tokenizer,
    output_csv="submission.csv",
    batch_size=32,
    max_length=512,
)


04/15/2026 01:38:37 - INFO - __main__ - 
04/15/2026 01:38:37 - INFO - __main__ - INFERENCE
04/15/2026 01:38:37 - INFO - __main__ - ================================================================================



04/15/2026 01:38:37 - INFO - __main__ - 
04/15/2026 01:38:37 - INFO - __main__ - INFERENCE
04/15/2026 01:38:37 - INFO - __main__ - ================================================================================



NameError: name 'trainer' is not defined

In [ ]:
# ============================================================================
# 6. (OPTIONAL) Upload trained model to Hugging Face Hub
# ============================================================================

def upload_to_hub(
    trainer,
    repo_id: str = "dzungpham/taskA-codebert-base",
    commit_message: str = "Upload fine-tuned CodeBERT model"
):
    """Upload the trained model to Hugging Face Hub."""
    try:
        logger.info(f"Uploading model to {repo_id}...")
        trainer.push_to_hub(
            repo_id=repo_id,
            commit_message=commit_message
        )
        logger.info(f"✅ Model uploaded to: https://huggingface.co/{repo_id}")
    except Exception as e:
        logger.warning(f"Could not upload to Hub: {e}")


# Uncomment the line below to upload the model to HF Hub
# upload_to_hub(trainer, repo_id="your-username/taskA-codebert-base")